In [2]:
import requests
import lxml.html
from pypdf import PdfReader
import pandas as pd
from google import genai
from pathlib import Path
from google.genai import types
import os
import us
from datetime import datetime

from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional, TypedDict, get_type_hints

In [3]:
# from ingest_sc_cases import case_df
# from ingest_sc_judges import judge_pd

In [4]:
case_df = pd.read_csv("../data/cases_scdb.csv")
judge_pd = pd.read_csv("../data/judges_slri.csv")

In [5]:
# Cases with no opinion documents
case_df[~case_df["opinion_link"].str.contains("statecourt", na=False)]

,Unnamed: 0,docket_no,title,state,date,type,pending,opinion_link
12,12,2026-00384,Williams v. Board of Elections of the State of...,New York,"March 19, 2026",Voting Rights and Elections,False,NaN
37,37,20220896,State v. Andrus,Utah,"August 7, 2025",Criminal Law,False,NaN
49,49,S-1-SC-40715,Franklin v. Martinez,New Mexico,"January 26, 2026",Criminal Law,False,NaN
101,101,22 OC 00022 1 B,Persaud-Zamora v. Cegavske,Nevada,"April 26, 2022",Voting Rights and Elections,False,NaN
105,105,EF2025-2536,Texas v. Bruck,Texas,"October 31, 2025",Reproductive Rights,False,NaN
107,107,2025AP002121,Voces de la Frontera v. Gerber,Wisconsin,"September 18, 2025",Government Structure,False,NaN
124,124,PR-123179,McVay v. Cockroft,Oklahoma,"July 28, 2025",Voting Rights and Elections,False,NaN
156,156,15-25-00093-CV,State v. City of San Antonio,Texas,"June 19, 2025",Reproductive Rights,False,NaN
210,210,CV01-23-14744,Adkins v. State,Idaho,"April 11, 2025",Reproductive Rights,False,NaN
232,232,SJC-13666,Gotay v. Creen,Massachusetts,"March 21, 2025",Civil Due Process,False,NaN


In [6]:
# Manually enumerate rights here, but
# programmatically create dataframe columns with these names later
class RightsDict(TypedDict):
    environment: str = Field(
        description="The effect of the court's decision on environmental rights"
    )
    consumers: str = Field(description="The effect of the court's decision on consumer rights")
    reproductive_rights: str = Field(
        description="The effect of the court's decision on reproductive rights"
    )
    democratic_norms: str = Field(
        description="The effect of the court's decision on democratic norms"
    )
    free_press: str = Field(
        description="The effect of the court's decision on the right to free press"
    )
    public_health: str = Field(description="The effect of the court's decision on public health")
    separation_church_state: str = Field(
        description="The effect of the court's decision on separation of church and state"
    )
    voting_access: str = Field(description="The effect of the court's decision on voting rights")
    public_education: str = Field(
        description="The effect of the court's decision on the right to public education"
    )
    free_speech: str = Field(
        description="The effect of the court's decision on the right to free speech"
    )
    privacy: str = Field(description="The effect of the court's decision on privacy rights")
    worker_rights: str = Field(description="The effect of the court's decision on worker rights")


class IndividualOpinion(BaseModel):
    judge_name: str = Field(description="The full name of the judge giving an opinion.")
    ruling: str = Field(
        description='"concur" or "dissent" or "other" based on how this judge ruled,'
    )
    description: str = Field(
        description="An extremely brief description of the judge's own opinion on the case."
    )


class Case(BaseModel):
    # title: str = Field(description="The title of the case.")
    issue_debate: str = Field(
        description='A phrase starting with "Whether" that summarizes '
        "the main issue being debated in the case"
    )
    plaintiff_argument: str = Field(
        description="Briefly, the plaintiff's stance on the debate issue"
    )
    plaintiff_pro_con: str = Field(
        description="Whether the plaintiff's argument is generally pro or con for the debate issue"
    )
    defendant_argument: str = Field(
        description="Briefly, the defendant's stance on the debate issue"
    )
    decision_outcome: str = Field(
        description="The court's final decision for the case whether "
        "they ruled with the plaintiff or not."
    )
    decision_winner: str = Field(
        description='The party that the court sided with ("plaintiff","defendant","other").'
    )

    rights_affected: RightsDict

    judge_opinions: List[IndividualOpinion]

In [7]:
def read_opinion(pdf_link: str, model_id: str, client: genai.Client, prompt: str):
    resp = requests.get(pdf_link).content

    genai_resp = client.models.generate_content(
        model=model_id,
        contents=[types.Part.from_bytes(data=resp, mime_type="application/pdf"), prompt],
        config={
            "response_mime_type": "application/json",
            "response_json_schema": Case.model_json_schema(),
        },
    )

    return Case.model_validate_json(genai_resp.text)

In [8]:
def analyze_state_cases(
    case_df: pd.DataFrame, judge_df: pd.DataFrame, prompt_start: str, client_info: dict
):
    prompt = prompt_start + "$DATAFRAME$" + judge_df.to_markdown(index=False)
    num_cases = len(case_df)
    model_id = client_info["model_id"]
    client = client_info["client"]

    file_dic = {}
    for i in range(num_cases):
        print(f"Querying case {i}")
        docket_no = case_df.iloc[i]["docket_no"].replace(" ", "-")
        state = case_df.iloc[i]["state"]
        date = str(datetime.strptime(case_df.iloc[i]["date"], "%B %d, %Y"))[:10].replace("-", "/")
        pdf_link = case_df.iloc[i]["opinion_link"]

        case_ref = "_".join([docket_no, state, date])

        opinion_resp = read_opinion(pdf_link, model_id, client, prompt)

        file_dic[case_ref] = {}
        file_dic[case_ref]["pdf_link"] = pdf_link
        file_dic[case_ref]["response"] = opinion_resp.model_dump()

    return file_dic

In [9]:
def apply_model(
    case_df: pd.DataFrame,
    judge_df: pd.DataFrame,
    prompt_path: str,
    model_id: str = "gemini-2.5-flash",
):
    with open(prompt_path, "r") as prompt_file:
        prompt_start = prompt_file.read()

    load_dotenv(find_dotenv())

    gemini_key = os.getenv("GEMINI_API_KEY")
    client_info = {"model_id": model_id, "client": genai.Client(api_key=gemini_key)}

    case_dic = analyze_state_cases(case_df, judge_df, prompt_start, client_info)

    return case_dic

In [10]:
def state_opinion_table(case_dic: dict):
    opinion_table = {"case_id": [], "name": [], "opinion": [], "ruling": []}

    for key in case_dic.keys():
        opinions = case_dic[key]["response"]
        num_opinions = len(opinions["judge_opinions"])

        for i in range(num_opinions):
            opinion_table["case_id"].append(key)

            judge_name = opinions["judge_opinions"][i]["judge_name"]
            opinion_table["name"].append(judge_name)

            description = opinions["judge_opinions"][i]["description"]
            opinion_table["opinion"].append(description)

            ruling = opinions["judge_opinions"][i]["ruling"]
            opinion_table["ruling"].append(ruling)

    return pd.DataFrame(opinion_table)

In [11]:
def state_case_table(case_df: pd.DataFrame, case_dic: dict):
    case_table = {
        "case_id": [],
        "docket_no": [],
        "title": [],
        "state": [],
        "date": [],
        "type": [],
        "description": [],
        "plaintiff_argument": [],
        "plaintiff_pro_con": [],
        "defendant_argument": [],
        "decision_outcome": [],
        "decision_winner": [],
        # "decision_status": [],
    }
    rights_enumerated_list = list(get_type_hints(RightsDict).keys())
    rights_enumerated_dict = {right: [] for right in rights_enumerated_list}
    case_table = case_table | rights_enumerated_dict

    num_cases = len(case_df)

    for i in range(num_cases):
        docket_no = case_df.iloc[i]["docket_no"].replace(" ", "-")
        case_table["docket_no"].append(docket_no)
        state = case_df.iloc[i]["state"]
        case_table["state"].append(state)
        date = str(datetime.strptime(case_df.iloc[i]["date"], "%B %d, %Y"))[:10].replace("-", "/")
        case_table["date"].append(date)

        title = case_df.iloc[i]["title"]
        case_table["title"].append(title)
        type = case_df.iloc[i]["type"]
        case_table["type"].append(type)

        case_id = "_".join([docket_no, state, date])
        case_table["case_id"].append(case_id)

        response = case_dic[case_id]["response"]
        case_table["description"].append(response["issue_debate"])
        case_table["plaintiff_argument"].append(response["plaintiff_argument"])
        case_table["plaintiff_pro_con"].append(response["plaintiff_pro_con"])
        case_table["defendant_argument"].append(response["defendant_argument"])
        case_table["decision_outcome"].append(response["decision_outcome"])
        case_table["decision_winner"].append(response["decision_winner"])

        rights_affected = response["rights_affected"]
        for right in rights_enumerated_list:
            case_table[right].append(rights_affected[right])

        # status = ~case_df.iloc[i]["pending"]  # Negate, "not pending" means "decided"
        # case_table["decision_status"].append(status)

    return pd.DataFrame(case_table)

In [12]:
def produce_tables(
    case_df: pd.DataFrame,
    judge_df: pd.DataFrame,
    prompt_path: str,
    model_id: str = "gemini-2.5-flash",
):
    states = case_df["state"].sort_values().unique()

    opinions = []
    cases = []

    for state in states:
        state_cases = case_df[case_df["state"] == state]

        state_abbr = str(us.states.lookup(state).abbr)
        state_judges = judge_df[judge_df["state"] == state_abbr]

        case_dic = apply_model(state_cases, state_judges, prompt_path, model_id)

        opinions.append(state_opinion_table(case_dic))
        cases.append(state_case_table(state_cases, case_dic))

    rd = {"opinion_table": pd.concat(opinions), "case_table": pd.concat(cases)}

    # Also return JSON metadata on this LLM batch run
    with open(prompt_path, "r") as prompt_file:
        prompt_start = prompt_file.read()
    llm_run_metadata = {
        "timestamp": datetime.today(),
        "model_id": model_id,
        "cases_processed": rd["case_table"]["case_id"].tolist(),
        "prompt_start": prompt_start,
    }

    return rd, llm_run_metadata

In [13]:
mon_cases = case_df[case_df["state"] == "Montana"].head(2)
docket_no_held = "DA 23-0575"  # Specifically an Environment case
held_case = case_df[case_df["docket_no"] == docket_no_held]
mon_cases = pd.concat([case_df[case_df["docket_no"] == docket_no_held], mon_cases])

mon_judges = judge_pd[judge_pd["state"] == "MT"]

In [14]:
mon_opinions, llm_run_metadata = produce_tables(
    mon_cases, mon_judges, "prompt.txt", "gemini-3-flash-preview"
)
# Ingestion script will save llm_run_metadata as JSON, appending with each run

Querying case 0
Querying case 1
Querying case 2


In [24]:
mon_opinions["opinion_table"]["case_id"].str.split("_", n=2, expand=True)

,0,1,2
0,DA-23-0575,Montana,2024/12/18
1,DA-23-0575,Montana,2024/12/18
2,DA-23-0575,Montana,2024/12/18
3,DA-23-0575,Montana,2024/12/18
4,DA-23-0575,Montana,2024/12/18
5,25-0139,Montana,2026/04/14
6,25-0139,Montana,2026/04/14
7,25-0139,Montana,2026/04/14
8,25-0139,Montana,2026/04/14
9,25-0139,Montana,2026/04/14


In [16]:
display(mon_opinions["opinion_table"])
display(mon_opinions["case_table"])

,case_id,name,opinion,ruling
0,DA-23-0575_Montana_2024/12/18,James Jeremiah Shea,Concurred with majority opinion,concur
1,DA-23-0575_Montana_2024/12/18,Beth Baker,Concurred with majority opinion,concur
2,DA-23-0575_Montana_2024/12/18,Ingrid Gustafson,Concurred with majority opinion,concur
3,DA-23-0575_Montana_2024/12/18,Laurie McKinnon,Concurred with majority opinion,concur
4,DA-23-0575_Montana_2024/12/18,James A. Rice,"Dissented, argued that the plaintiffs failed t...",dissent
5,25-0139_Montana_2026/04/14,Laurie McKinnon,"Concurred, argued that state policies likely v...",concur
6,25-0139_Montana_2026/04/14,James Jeremiah Shea,Concurred with majority opinion.,concur
7,25-0139_Montana_2026/04/14,Ingrid Gustafson,Concurred with majority opinion.,concur
8,25-0139_Montana_2026/04/14,Katherine Biidegaray,Concurred with majority opinion.,concur
9,25-0139_Montana_2026/04/14,Beth Baker,"Concurred, argued that the right to individual...",concur


,case_id,docket_no,title,state,date,type,description,plaintiff_argument,plaintiff_pro_con,defendant_argument,...,reproductive_rights,democratic_norms,free_press,public_health,separation_church_state,voting_access,public_education,free_speech,privacy,worker_rights
0,DA-23-0575_Montana_2024/12/18,DA-23-0575,Held v. Montana,Montana,2024/12/18,Environment,Whether the Montana Constitution's guarantee o...,The plaintiffs argued that the state's refusal...,pro,The defendants argued that the plaintiffs lack...,...,NA,protected,NA,protected,NA,NA,NA,NA,protected,NA
1,25-0139_Montana_2026/04/14,25-0139,Kalarchik v. State,Montana,2026/04/14,Civil Rights,Whether Montana state policies restricting the...,The policies discriminate based on sex and tra...,pro,"The policies apply equally to all, serve a com...",...,NA,NA,NA,NA,NA,NA,NA,NA,protected,NA
2,OP-25-0858_Montana_2026/02/27,OP-25-0858,Kendrick v. Knudsen,Montana,2026/02/27,Voting Rights and Elections,Whether Ballot Issue 8 (BI-8) violates the sep...,"BI-8 is a single, comprehensive proposal that ...",pro,BI-8 violates the separate-vote requirement be...,...,NA,protected,NA,NA,NA,protected,NA,NA,NA,NA


In [17]:
# Good sign: Stopping state law that discriminated against trans people
# flagged "privacy: protected"
mon_opinions["case_table"].loc[1, :]

case_id                                           25-0139_Montana_2026/04/14
docket_no                                                            25-0139
title                                                     Kalarchik v. State
state                                                                Montana
date                                                              2026/04/14
type                                                            Civil Rights
description                Whether Montana state policies restricting the...
plaintiff_argument         The policies discriminate based on sex and tra...
plaintiff_pro_con                                                        pro
defendant_argument         The policies apply equally to all, serve a com...
decision_outcome           The court affirmed the preliminary injunction ...
decision_winner                                                    plaintiff
environment                                                               NA